# Coupling Groups, Copulas and Variable Reordering

This notebook demonstrates how PAL manages dependencies between stochastic variables using:

1. **Coupling Groups** — automatic tracking of related variables
2. **Copulas** — creating dependency structures between variables
3. **Variable Reordering** — how copulas preserve marginal distributions while introducing correlation

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pal import config, copulas, distributions, set_random_seed

config.n_sims = 10_000
set_random_seed(42)

## 1. Coupling Groups

Every `StochasticScalar` belongs to a **coupling group** — a set of variables linked by computation. When you combine variables in arithmetic, their coupling groups **merge automatically**.

In [ ]:
motor = distributions.LogNormal(mu=14, sigma=0.5).generate()
prop = distributions.LogNormal(mu=15, sigma=0.8).generate()

print("Each variable starts in its own coupling group:")
print(f"  Same group? {motor.coupled_variable_group is prop.coupled_variable_group}")

total = motor + prop

print("\nAfter total = motor + prop:")
print(f"  Same group? {motor.coupled_variable_group is prop.coupled_variable_group}")
print(f"  Group size: {len(motor.coupled_variable_group)}")

motor_expenses = motor * 1.1  # Derived variables join automatically
print("\nAfter motor_expenses = motor * 1.1:")
print(f"  Group size: {len(motor.coupled_variable_group)}")

### Why Coupling Groups Matter

When a copula reorders one variable, **all coupled variables are reordered identically**, preserving their mathematical relationships.

In [ ]:
set_random_seed(42)
loss_a = distributions.LogNormal(mu=14, sigma=0.5).generate()
loss_b = distributions.LogNormal(mu=15, sigma=0.8).generate()
loss_a_expenses = loss_a * 1.15  # 15% expense loading

print("Before copula:")
ratio_before = loss_a_expenses.values[:5] / loss_a.values[:5]
print(f"  Ratio: {ratio_before}")

copulas.GaussianCopula([[1.0, 0.7], [0.7, 1.0]]).apply([loss_a, loss_b])

print("\nAfter copula reordering:")
ratio_after = loss_a_expenses.values[:5] / loss_a.values[:5]
print(f"  Ratio: {ratio_after}")
print("\nThe 1.15x relationship is preserved!")

## 2. Copulas — Creating Dependency Structures

PAL provides several copula families. Each produces different patterns of dependence, especially in the tails.

| Family | Class | Tail Dependence |
|--------|-------|-----------------|
| Gaussian | `GaussianCopula` | None |
| Student's T | `StudentsTCopula` | Symmetric |
| Gumbel | `GumbelCopula` | Upper |
| Clayton | `ClaytonCopula` | Lower |
| Frank | `FrankCopula` | None |

In [ ]:
def make_pair():
    """Generate a fresh pair of independent LogNormal variables."""
    set_random_seed(42)
    x = distributions.LogNormal(mu=10, sigma=1.0).generate()
    y = distributions.LogNormal(mu=10, sigma=1.0).generate()
    return x, y


def rank_corr(a, b):
    """Compute Spearman rank correlation."""
    return np.corrcoef(a.ranks.values, b.ranks.values)[0, 1]


# Independent
x_indep, y_indep = make_pair()
print(f"Independent:       {rank_corr(x_indep, y_indep):.4f}")

# Gaussian
x_gauss, y_gauss = make_pair()
copulas.GaussianCopula([[1, 0.8], [0.8, 1]]).apply([x_gauss, y_gauss])
print(f"Gaussian (rho=0.8): {rank_corr(x_gauss, y_gauss):.4f}")

# Gumbel
x_gumbel, y_gumbel = make_pair()
copulas.GumbelCopula(theta=3.0, n=2).apply([x_gumbel, y_gumbel])
print(f"Gumbel (theta=3.0): {rank_corr(x_gumbel, y_gumbel):.4f}")

# Clayton
x_clay, y_clay = make_pair()
copulas.ClaytonCopula(theta=4.0, n=2).apply([x_clay, y_clay])
print(f"Clayton (theta=4.0): {rank_corr(x_clay, y_clay):.4f}")

# Student's T
x_t, y_t = make_pair()
copulas.StudentsTCopula([[1, 0.7], [0.7, 1]], dof=3).apply([x_t, y_t])
print(f"Student's T (rho=0.7, dof=3): {rank_corr(x_t, y_t):.4f}")

### Scatter Plots — Rank Space

Plotting in **rank space** (ranks instead of raw values) isolates the dependency structure from the marginal distributions.

In [ ]:
fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        "Independent",
        "Gaussian (\u03c1=0.8)",
        "Gumbel (\u03b8=3.0)",
        "Clayton (\u03b8=4.0)",
        "Student's T (\u03c1=0.7, \u03bd=3)",
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

scatter_kw = {"mode": "markers", "marker": {"size": 2, "opacity": 0.3}}
pairs = [
    (x_indep, y_indep, "Independent", 1, 1),
    (x_gauss, y_gauss, "Gaussian", 1, 2),
    (x_gumbel, y_gumbel, "Gumbel", 1, 3),
    (x_clay, y_clay, "Clayton", 2, 1),
    (x_t, y_t, "Student's T", 2, 2),
]

for xv, yv, name, row, col in pairs:
    fig.add_trace(
        go.Scattergl(
            x=xv.ranks.values.tolist(),
            y=yv.ranks.values.tolist(),
            name=name,
            **scatter_kw,
        ),
        row=row,
        col=col,
    )

fig.update_layout(
    title_text="Copula Dependency Structures (Rank Space)",
    height=700,
    width=1000,
    showlegend=False,
)
for _, _, _, row, col in pairs:
    fig.update_xaxes(title_text="X Rank", row=row, col=col)
    fig.update_yaxes(title_text="Y Rank", row=row, col=col)

fig.show()

**What to observe:**
- **Independent**: uniform scatter, no pattern
- **Gaussian**: elliptical concentration along the diagonal
- **Gumbel**: strong clustering in the **upper-right** corner (upper tail dependence)
- **Clayton**: strong clustering in the **lower-left** corner (lower tail dependence)
- **Student's T**: like Gaussian but with heavier concentration in **both** corners

## 3. Variable Reordering

`copula.apply()` works by **reordering** simulations to match the copula's rank structure. The marginal distributions are **exactly preserved** — the same values exist, just in a different order.

In [ ]:
set_random_seed(42)
var_x = distributions.Normal(0, 1).generate()
var_y = distributions.Normal(0, 1).generate()

sorted_x = np.sort(var_x.values)
sorted_y = np.sort(var_y.values)

print(f"Before: X mean={var_x.mean():.4f}, std={var_x.std():.4f}")
print(f"Before: Y mean={var_y.mean():.4f}, std={var_y.std():.4f}")

copulas.GaussianCopula([[1, 0.9], [0.9, 1]]).apply([var_x, var_y])

print(f"\nAfter:  X mean={var_x.mean():.4f}, std={var_x.std():.4f}")
print(f"After:  Y mean={var_y.mean():.4f}, std={var_y.std():.4f}")

print(f"\nX values unchanged? {np.allclose(np.sort(var_x.values), sorted_x)}")
print(f"Y values unchanged? {np.allclose(np.sort(var_y.values), sorted_y)}")
print(f"Rank correlation:   {rank_corr(var_x, var_y):.4f}")

### Reordering Across a Chain of Derived Variables

Coupling groups enable **transitive reordering**. When a copula is applied to a base variable, every derived variable in the chain is reordered together.

In [ ]:
set_random_seed(42)

base_loss = distributions.LogNormal(mu=14, sigma=0.5).generate()
gross_loss = base_loss * 1.0
expense_loaded = gross_loss * 1.10
tax = expense_loaded * 0.21
net_loss = expense_loaded - tax

print(f"Coupling group size: {len(base_loss.coupled_variable_group)}")

cat_loss = distributions.LogNormal(mu=16, sigma=1.2).generate()
copulas.GumbelCopula(theta=1.5, n=2).apply([base_loss, cat_loss])

print("\nAfter copula between base_loss and cat_loss:")
print(f"  gross/base ratio:    {gross_loss.values[:3] / base_loss.values[:3]}")
print(f"  expense/gross ratio: {expense_loaded.values[:3] / gross_loss.values[:3]}")
print(f"  tax/expense ratio:   {tax.values[:3] / expense_loaded.values[:3]}")
print("\nAll ratios preserved!")

## 4. Multivariate Copulas

Elliptical copulas naturally extend to any number of dimensions. Here we correlate 4 lines of business.

In [ ]:
set_random_seed(42)

lobs = {
    "Motor": distributions.LogNormal(mu=14, sigma=0.4).generate(),
    "Property": distributions.LogNormal(mu=15, sigma=0.6).generate(),
    "Liability": distributions.LogNormal(mu=13, sigma=0.5).generate(),
    "Marine": distributions.LogNormal(mu=12, sigma=0.7).generate(),
}

corr_matrix = [
    [1.0, 0.6, 0.3, 0.2],
    [0.6, 1.0, 0.4, 0.3],
    [0.3, 0.4, 1.0, 0.5],
    [0.2, 0.3, 0.5, 1.0],
]

copulas.GaussianCopula(corr_matrix).apply(list(lobs.values()))

names = list(lobs.keys())
ranks = np.array([lobs[n].ranks.values for n in names])
result = np.corrcoef(ranks)

print("Resulting rank correlation matrix:")
print(f"{'':>12}", end="")
for n in names:
    print(f"{n:>12}", end="")
print()
for i, n in enumerate(names):
    print(f"{n:>12}", end="")
    for j in range(len(names)):
        print(f"{result[i, j]:>12.3f}", end="")
    print()

print("\nInput correlation matrix:")
for row in corr_matrix:
    print("  ", [f"{x:.1f}" for x in row])

## 5. `generate()` vs `apply()`

- **`generate()`** — creates new correlated **uniform** samples
- **`apply()`** — reorders **existing** variables to match the copula structure

In most actuarial models, you'll use `apply()` because variables have already been generated from their marginal distributions.

In [ ]:
set_random_seed(42)

# generate() returns uniform samples
samples = copulas.GaussianCopula([[1, 0.8], [0.8, 1]]).generate()
print("generate() returns uniform samples:")
print(f"  Mean[0]: {samples[0].mean():.4f}  (should be ~0.5)")
print(f"  Mean[1]: {samples[1].mean():.4f}  (should be ~0.5)")
print(f"  Rank corr: {rank_corr(samples[0], samples[1]):.4f}")

# apply() reorders existing variables
set_random_seed(42)
v1 = distributions.Gamma(alpha=5, theta=1000).generate()
v2 = distributions.Pareto(shape=2, scale=10000).generate()
print("\napply() reorders existing variables:")
print(f"  Before: V1 mean={v1.mean():.0f}, V2 mean={v2.mean():.0f}")
copulas.GaussianCopula([[1, 0.8], [0.8, 1]]).apply([v1, v2])
print(f"  After:  V1 mean={v1.mean():.0f}, V2 mean={v2.mean():.0f}")
print(f"  Rank corr: {rank_corr(v1, v2):.4f}")
print("\nMeans unchanged — only the ordering changed!")